# Лабораторная работа №3
## Изображения и видео

Дата выдачи: 28.04.2026
Оценка на 5: 12.05.2026
Оценка на 4: 25.05.2026

“Воровство разметки”. Не всегда есть возможность размечать фото или видео самостоятельно, но случается, когда под рукой визуализация чужого результата разметки. Задача извлечь существующую разметку объектов из предоставленного видео output.mp4 в формат COCO. Оригинал видео без bbox находится в файле input.mp4

Проверить качество извлечения разметки можно сравнивая с оригиналом разметки
Результат работы - оценка качества извлеченной разметки по IoU и отрисовка на новом видео именно вашей извлеченной разметки методами opencv.


### Задачи
- Скачать предоставленное демонстрационное видео и выполнить декодирование в последовательность кадров.
- Использовать публичный компьютерно-зрительный API для получения координат bounding box’ов на каждом кадре. Попробовать несколько методов нахождения bbox и предоставить их сравнительный анализ
-Оценить качество извлеченной разметки сравнивая с оригиналом по IoU
- Сформировать структурированный датасет с аннотациями в одном из стандартных форматов: COCO
- Обучить детектор объектов на основе готовой архитектуры из torchvision.models.detection (например, Faster R-CNN или RetinaNet).
- Оценить качество модели с использованием метрики mAP (mean Average Precision) на валидационной выборке

### Критерии оценивания (100 баллов)
- Извлечение и форматирование разметки (до 35 баллов).
- Корректно извлечены кадры из видео и получены bounding box’ы с помощью публичного CV-API. Аннотации сохранены в формате COCO и соответствуют стандарту. Проведено сравнение с оригиналом по IoU
- Обучение модели и контроль переобучения (до 25 баллов).
- (До)обучена модель детекции на основе torchvision.models.detection. Использована валидационная выборка, представлены графики потерь, применены меры против переобучения (например, early stopping).
- Метрики и их интерпретация (до 15 баллов).
- Качество оценено по mAP. Приведен анализ результатов: какие объекты детектируются хорошо/плохо
- Визуализация результатов (до 15 баллов).
- Показаны примеры кадров с предсказанными bounding box’ами — как успешные, так и ошибочные. Визуализации читаемы и подписаны.
Оформление и обсуждение ограничений (до 10 баллов)


## 1. Пытаемся извлечь разметку из видео и пытаемся получить разметку

### Открываем видео и пробуем разбить его на кадры с шагом 5:

In [1]:
import cv2
import os
import json
import numpy as np

def video_to_frames(video_path, output_folder,step=5):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print("Ошибка, не удалось открыть видеофайл")
        return

    frame_count = 0
    
    while True:
        # ret (True/False — удалось ли считать кадр) и сам frame (матрица пикселей)
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % step != 0:
            frame_count += 1
            continue
        frame_filename = os.path.join(output_folder, f"frame_{frame_count:04d}.jpg")
        cv2.imwrite(frame_filename, frame)
        
        frame_count += 1
    
    cap.release()
    print(f"Готово! Декодировано и сохранено кадров: {frame_count}")

video_to_frames("../data/input.mp4", "../data/extracted_frames")

Готово! Декодировано и сохранено кадров: 301


### Метод 1: пробуем через разности собрать разметку:

In [ ]:
def extract_frames_with_diff(input_video, output_video, clean_frames_folder, debug_folder, step=5):
    os.makedirs(clean_frames_folder, exist_ok=True)
    os.makedirs(debug_folder, exist_ok=True)
    
    cap_in = cv2.VideoCapture(input_video)
    cap_out = cv2.VideoCapture(output_video)
    
    if not cap_in.isOpened() or not cap_out.isOpened():
        print("Ошибка: не удалось открыть одно из видео!")
        return

    frame_count = 0
    saved_count = 0
    annotations = {}

    while True:
        ret_in, frame_in = cap_in.read()
        ret_out, frame_out = cap_out.read()
        
        if not ret_in or not ret_out:
            break
            
        if frame_count % step != 0:
            frame_count += 1
            continue

        diff = cv2.absdiff(frame_out, frame_in)
        gray_diff = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray_diff, 40, 255, cv2.THRESH_BINARY)
        kernel = np.ones((3, 3), np.uint8)
        closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(closed, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        
        bboxes_in_frame = []
        debug_frame = frame_in.copy()
        
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            area = w * h
            
            if 500 < area < 150000 and 10 < w < 400 and 10 < h < 400:
                # проверка на пропорции, похожие на рамки
                aspect_ratio = w / float(h)
                if 0.2 < aspect_ratio < 5.0:
                    bboxes_in_frame.append([x, y, w, h])
                    cv2.rectangle(debug_frame, (x, y), (x + w, y + h), (0, 0, 255), 2)


        frame_filename = f"frame_{frame_count:04d}.jpg"
        
        cv2.imwrite(os.path.join(clean_frames_folder, frame_filename), frame_in)
        cv2.imwrite(os.path.join(debug_folder, frame_filename), debug_frame)
        
        annotations[frame_filename] = bboxes_in_frame
        
        frame_count += 1
        saved_count += 1
    
    cap_in.release()
    cap_out.release()
    
    with open("../data/extracted_annotations_method_1.json", "w") as f:
        json.dump(annotations, f, indent=4)
        
    print(f"Готово Обработано кадров: {frame_count}, сохранено: {saved_count}")


extract_frames_with_diff(
    input_video="../data/input.mp4", 
    output_video="../data/output.mp4", 
    clean_frames_folder="../data/extracted_frames",
    debug_folder="../data/debug_frames_method_1", 
    step=5
)

Готово! Обработано кадров: 301, сохранено: 61


### Метод 2: Детекция граней и поиск прямоугольников
В чем суть: Рамка (bbox) — это идеальный прямоугольник. У нее четкие и резкие границы. Мы можем использовать детектор границ Кенни (cv2.Canny), чтобы найти все резкие переходы цвета (линии), а затем заставить OpenCV искать среди них только те объекты, у которых ровно 4 угла.

In [ ]:
def extract_frames_with_diff(input_video, output_video, clean_frames_folder, debug_folder, step=5):
    os.makedirs(clean_frames_folder, exist_ok=True)
    os.makedirs(debug_folder, exist_ok=True)
    
    cap_in = cv2.VideoCapture(input_video)
    cap_out = cv2.VideoCapture(output_video)
    
    if not cap_in.isOpened() or not cap_out.isOpened():
        print("Ошибка: не удалось открыть одно из видео!")
        return

    frame_count = 0
    saved_count = 0
    annotations = {}

    while True:
        ret_in, frame_in = cap_in.read()
        ret_out, frame_out = cap_out.read()
        
        if not ret_in or not ret_out:
            break
            
        if frame_count % step != 0:
            frame_count += 1
            continue

        diff = cv2.absdiff(frame_out, frame_in)
        gray_diff = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        # убираем шум
        blurred = cv2.GaussianBlur(gray_diff, (5, 5), 0)
        # ищем резкие границы
        edges = cv2.Canny(blurred, 50, 150)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        closed_edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(closed_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        bboxes_in_frame = []
        debug_frame = frame_in.copy()
        
        for contour in contours:
            epsilon = 0.02 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True)
            if 4 <= len(approx) <= 6:
                x, y, w, h = cv2.boundingRect(contour)
                area = w * h
                
                
                if 500 < area < 150000:
                    bboxes_in_frame.append([x, y, w, h])
                    cv2.rectangle(debug_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        

        frame_filename = f"frame_{frame_count:04d}.jpg"
        
        cv2.imwrite(os.path.join(clean_frames_folder, frame_filename), frame_in)
        cv2.imwrite(os.path.join(debug_folder, frame_filename), debug_frame)
        
        annotations[frame_filename] = bboxes_in_frame
        
        frame_count += 1
        saved_count += 1
    
    cap_in.release()
    cap_out.release()
    
    with open("../data/extracted_annotations_method_2.json", "w") as f:
        json.dump(annotations, f, indent=4)
        
    print(f"Готово Обработано кадров: {frame_count}, сохранено: {saved_count}")


extract_frames_with_diff(
    input_video="../data/input.mp4", 
    output_video="../data/output.mp4", 
    clean_frames_folder="../data/extracted_frames",
    debug_folder="../data/debug_frames_method_2", 
    step=5
)

Готово! Обработано кадров: 301, сохранено: 61


### Способ 3: разметка с помощью yolo:

In [ ]:
import cv2
import os
import json
from ultralytics import YOLO

def extract_bboxes_with_yolo(frames_folder, debug_folder):
    os.makedirs(debug_folder, exist_ok=True)
    model = YOLO("yolov8m.pt")
    
    annotations = {}
    frame_files = sorted([f for f in os.listdir(frames_folder) if f.endswith('.jpg')])
    
    # 2: car, 3: motorcycle, 5: bus, 7: truck
    target_classes = [2, 3, 5, 7] 

    for frame_name in frame_files:
        frame_path = os.path.join(frames_folder, frame_name)
        frame = cv2.imread(frame_path)
        results = model(frame, verbose=False)
        
        bboxes_in_frame = []
        debug_frame = frame.copy()
        for box in results[0].boxes:
            class_id = int(box.cls[0].item())
            if class_id in target_classes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                
                
                x = int(x1)
                y = int(y1)
                w = int(x2 - x1)
                h = int(y2 - y1)
                
                # увереность
                confidence = float(box.conf[0].item())
                if confidence > 0.3:
                    bboxes_in_frame.append([x, y, w, h])
                    cv2.rectangle(debug_frame, (x, y), (x + w, y + h), (255, 0, 255), 2)
                    label = model.names[class_id]
                    cv2.putText(debug_frame, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 2)

        cv2.imwrite(os.path.join(debug_folder, frame_name), debug_frame)
        annotations[frame_name] = bboxes_in_frame
        
        print(f"Обработан кадр: {frame_name}, найдено объектов: {len(bboxes_in_frame)}")
    with open("../data/yolo_annotations.json", "w") as f:
        json.dump(annotations, f, indent=4)
        
    print(f"Готово! Разметка YOLO сохранена в yolo_annotations.json")
    print(f"Проверьте картинки в папке: {debug_folder}")

# Запуск
extract_bboxes_with_yolo(
    frames_folder="../data/extracted_frames", # отсюда берем чистые кадры
    debug_folder="../data/debug_yolo"         # сюда сохраняем картинки с рамками
)

Обработан кадр: frame_0000.jpg, найдено объектов: 10
Обработан кадр: frame_0005.jpg, найдено объектов: 8
Обработан кадр: frame_0010.jpg, найдено объектов: 8
Обработан кадр: frame_0015.jpg, найдено объектов: 8
Обработан кадр: frame_0020.jpg, найдено объектов: 9
Обработан кадр: frame_0025.jpg, найдено объектов: 8
Обработан кадр: frame_0030.jpg, найдено объектов: 9
Обработан кадр: frame_0035.jpg, найдено объектов: 8
Обработан кадр: frame_0040.jpg, найдено объектов: 10
Обработан кадр: frame_0045.jpg, найдено объектов: 9
Обработан кадр: frame_0050.jpg, найдено объектов: 9
Обработан кадр: frame_0055.jpg, найдено объектов: 11
Обработан кадр: frame_0060.jpg, найдено объектов: 8
Обработан кадр: frame_0065.jpg, найдено объектов: 8
Обработан кадр: frame_0070.jpg, найдено объектов: 8
Обработан кадр: frame_0075.jpg, найдено объектов: 8
Обработан кадр: frame_0080.jpg, найдено объектов: 7
Обработан кадр: frame_0085.jpg, найдено объектов: 9
Обработан кадр: frame_0090.jpg, найдено объектов: 8
Обработан

### Сравниваем разметки способами 1,2 и yolo:

In [ ]:
import json

def calculate_iou(boxA, boxB):
    boxA_x1, boxA_y1, boxA_w, boxA_h = boxA
    boxA_x2, boxA_y2 = boxA_x1 + boxA_w, boxA_y1 + boxA_h
    
    boxB_x1, boxB_y1, boxB_w, boxB_h = boxB
    boxB_x2, boxB_y2 = boxB_x1 + boxB_w, boxB_y1 + boxB_h
    xA = max(boxA_x1, boxB_x1)
    yA = max(boxA_y1, boxB_y1)
    xB = min(boxA_x2, boxB_x2)
    yB = min(boxA_y2, boxB_y2)
    inter_width = max(0, xB - xA)
    inter_height = max(0, yB - yA)
    interArea = inter_width * inter_height
    if interArea == 0:
        return 0.0
    boxAArea = boxA_w * boxA_h
    boxBArea = boxB_w * boxB_h
    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

def evaluate_markup(original_json_path, predicted_json_path):
    with open(original_json_path, 'r') as f:
        original_data = json.load(f)
        
    with open(predicted_json_path, 'r') as f:
        predicted_data = json.load(f)

    frame_ious = []

    for frame_name, orig_boxes in original_data.items():
        pred_boxes = predicted_data.get(frame_name, [])
        
        if not orig_boxes and not pred_boxes:
            continue
        
        if orig_boxes and not pred_boxes:
             frame_ious.append(0.0)
             continue
             
        matched_ious = []
        for orig_box in orig_boxes:
            best_iou = 0.0
            
            for pred_box in pred_boxes:
                iou = calculate_iou(orig_box, pred_box)
                if iou > best_iou:
                    best_iou = iou
            
            matched_ious.append(best_iou)

        if matched_ious:
            mean_frame_iou = sum(matched_ious) / len(matched_ious)
            frame_ious.append(mean_frame_iou)

    final_mIoU = sum(frame_ious) / len(frame_ious) if frame_ious else 0
    print(f"=====================================")
    print(f"Средний IoU по всему видео: {final_mIoU:.4f} (или {final_mIoU * 100:.1f}%)")
    print(f"=====================================")
    
    return final_mIoU

evaluate_markup(
    original_json_path="../data/extracted_annotations_method_1.json", # метод 1 - вычитание
    predicted_json_path="../data/yolo_annotations.json"      # Разметка от YOLO
)
evaluate_markup(
    original_json_path="../data/extracted_annotations_method_2.json", # метод 2 - детекция граней
    predicted_json_path="../data/yolo_annotations.json"      # Разметка от YOLO
)
evaluate_markup(
    original_json_path="../data/extracted_annotations_method_1.json", # метод 1 - вычитание
    predicted_json_path="../data/extracted_annotations_method_2.json"      # метод 2 - детекция граней
)

Средний IoU по всему видео: 0.5982 (или 59.8%)
Средний IoU по всему видео: 0.6755 (или 67.5%)
Средний IoU по всему видео: 0.1113 (или 11.1%)


0.11125633707719472

#### Выводы по разметке:

- Хотя результат yolo vs метод 2 кажется наилучшим, но это неправда - в методе 2 очень мало размеченных машин, а в методе 1 сильно много наползающих друг на друга bbox-ов, поэтому выбор падает на yolo8n. Она размечает машины вблизи почти идеально, правда вдалеке работает плохо из-за сжатия картинок.

- Сравнительный анализ показал, что легкая модель yolo8n плохо справляется с нестандартным ракурсом камеры. При переходе на более глубокую модель YOLOv8m качество визуальной детекции стало практически идеальным (рамки плотно охватывают целевые объекты, учитываются даже частично перекрытые машины). Однако средний IoU с оригинальной XML-разметкой составил всего 25.7%. Это связано с тем, что в Ground Truth (XML) размечены микроскопические объекты на горизонте дороги, которые пропадают при ресайзе изображения внутри нейросети (False Negatives). Эти нулевые значения IoU сильно занижают общее среднее, несмотря на высокое качество детекции основных объектов.

### Парсим xml файл для получения эталонной разметки:

In [ ]:
import xml.etree.ElementTree as ET


def parse_xml_and_evaluate(xml_path, yolo_json_path, step=5):
    print("Читаем XML файл...")
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    original_annotations = {}
    for track in root.findall('.//track'):
        for box in track.findall('.//box'):
            frame_idx = int(box.get('frame'))
        
            if frame_idx % step != 0:
                continue
                
            frame_name = f"frame_{frame_idx:04d}.jpg"
            xtl = float(box.get('xtl', box.get('xmin', 0)))
            ytl = float(box.get('ytl', box.get('ymin', 0)))
            xbr = float(box.get('xbr', box.get('xmax', 0)))
            ybr = float(box.get('ybr', box.get('ymax', 0)))
            x = int(xtl)
            y = int(ytl)
            w = int(xbr - xtl)
            h = int(ybr - ytl)
            
            if frame_name not in original_annotations:
                original_annotations[frame_name] = []
            original_annotations[frame_name].append([x, y, w, h])

    print(f"Извлечена оригинальная разметка для {len(original_annotations)} кадров.")

    with open(yolo_json_path, 'r') as f:
        yolo_annotations = json.load(f)

    frame_ious = []
    
    for frame_name, orig_boxes in original_annotations.items():
        pred_boxes = yolo_annotations.get(frame_name, [])
        
        if not orig_boxes and not pred_boxes:
            continue
            
        if orig_boxes and not pred_boxes:
            frame_ious.append(0.0)
            continue
            
        matched_ious = []
        for orig_box in orig_boxes:
            best_iou = 0.0
            for pred_box in pred_boxes:
                iou = calculate_iou(orig_box, pred_box)
                if iou > best_iou:
                    best_iou = iou
            matched_ious.append(best_iou)

        if matched_ious:
            frame_ious.append(sum(matched_ious) / len(matched_ious))

    if not frame_ious:
        print("Ошибка: Не удалось сопоставить кадры. Проверь названия файлов или шаг (step).")
        return

    final_mIoU = sum(frame_ious) / len(frame_ious)
    
    print(f"=====================================")
    print(f"Реальный IoU (YOLO против Оригинала XML): {final_mIoU:.4f} (или {final_mIoU * 100:.1f}%)")
    print(f"=====================================")


parse_xml_and_evaluate(
    xml_path="../data/annotations.xml",
    yolo_json_path="../data/yolo_annotations.json",
    step=5
)
parse_xml_and_evaluate(
    xml_path="../data/annotations.xml",
    yolo_json_path="../data/extracted_annotations_method_1.json",
    step=5
)

Читаем XML файл...
Извлечена оригинальная разметка для 181 кадров.
Реальный IoU (YOLO против Оригинала XML): 0.2569 (или 25.7%)
Читаем XML файл...
Извлечена оригинальная разметка для 181 кадров.
Реальный IoU (YOLO против Оригинала XML): 0.2378 (или 23.8%)


### Визуализируем разметку yolo vs эталон:

Зеленый = эталон

Красный = разметка yolo

In [ ]:
def debug_markup_visualizer(xml_path, yolo_json_path, frames_folder, debug_folder, step=5):
    os.makedirs(debug_folder, exist_ok=True)
    
    print("Парсим XML с умной фильтрацией...")
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    original_annotations = {}
    valid_labels = ['car', 'bus', 'truck', 'vehicle']

    for track in root.findall('.//track'):
        label = track.get('label', '').lower()
        if label and label not in valid_labels:
            continue

        for box in track.findall('.//box'):
            if box.get('outside') == '1':
                continue
            if box.get('occluded') == '1':
                continue

            frame_idx = int(box.get('frame'))
            if frame_idx % step != 0:
                continue
                
            frame_name = f"frame_{frame_idx:04d}.jpg"
            
            xtl = float(box.get('xtl', box.get('xmin', 0)))
            ytl = float(box.get('ytl', box.get('ymin', 0)))
            xbr = float(box.get('xbr', box.get('xmax', 0)))
            ybr = float(box.get('ybr', box.get('ymax', 0)))
            
            x, y = int(xtl), int(ytl)
            w, h = int(xbr - xtl), int(ybr - ytl)
            
            if w * h < 400:
                continue
            
            if frame_name not in original_annotations:
                original_annotations[frame_name] = []
            original_annotations[frame_name].append([x, y, w, h])

    print(f"Отфильтровано кадров с разметкой: {len(original_annotations)}")

    with open(yolo_json_path, 'r') as f:
        yolo_annotations = json.load(f)

    frames_to_draw = list(original_annotations.keys())[:5]
    
    for frame_name in frames_to_draw:
        frame_path = os.path.join(frames_folder, frame_name)
        if not os.path.exists(frame_path):
            continue
            
        img = cv2.imread(frame_path)
        orig_boxes = original_annotations.get(frame_name, [])
        for box in orig_boxes:
            x, y, w, h = box
            cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(img, "XML", (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        yolo_boxes = yolo_annotations.get(frame_name, [])
        for box in yolo_boxes:
            x, y, w, h = box
            cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 255), 2)
            cv2.putText(img, "YOLO", (x, y+h+15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        cv2.imwrite(os.path.join(debug_folder, f"compare_{frame_name}"), img)
        
    print(f"Картинки для сравнения сохранены в папку {debug_folder}. Иди посмотри на них!")

debug_markup_visualizer(
    xml_path="../data/annotations.xml", 
    yolo_json_path="../data/yolo_annotations.json",
    frames_folder="../data/extracted_frames", 
    debug_folder="../data/debug_compare", 
    step=5
)

Парсим XML с умной фильтрацией...
Отфильтровано кадров с разметкой: 181
Картинки для сравнения сохранены в папку ../data/debug_compare. Иди посмотри на них!


### Делаем COCO разметку для дальнейшего обучения моделей

In [ ]:
import random
import shutil


def create_coco_dataset(frames_folder, annotations_path, output_dataset_dir, split_ratio=0.8):
    train_dir = os.path.join(output_dataset_dir, "train")
    val_dir = os.path.join(output_dataset_dir, "val")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    print("Загрузка разметки...")
    with open(annotations_path, 'r') as f:
        yolo_data = json.load(f)

    all_frames = list(yolo_data.keys())
    random.seed(42)
    random.shuffle(all_frames)

    split_index = int(len(all_frames) * split_ratio)
    train_frames = all_frames[:split_index]
    val_frames = all_frames[split_index:]

    print(f"Всего кадров: {len(all_frames)}. На обучение (Train): {len(train_frames)}, На проверку (Val): {len(val_frames)}")

    def get_coco_template():
        return {
            "info": {"description": "Vehicle Detection Dataset"},
            "images": [],
            "annotations": [],
            "categories": [{"id": 1, "name": "vehicle", "supercategory": "none"}]
        }

    coco_train = get_coco_template()
    coco_val = get_coco_template()

    def process_split(frames_list, coco_dict, dest_folder):
        annotation_id = 1
        
        for img_id, frame_name in enumerate(frames_list, start=1):
            src_path = os.path.join(frames_folder, frame_name)
            dst_path = os.path.join(dest_folder, frame_name)
            if not os.path.exists(src_path):
                continue
                
            shutil.copy(src_path, dst_path)

            img = cv2.imread(dst_path)
            height, width = img.shape[:2]

            coco_dict["images"].append({
                "id": img_id,
                "file_name": frame_name,
                "width": width,
                "height": height
            })

            bboxes = yolo_data.get(frame_name, [])
            for bbox in bboxes:
                x, y, w, h = bbox
                area = w * h
                
                coco_dict["annotations"].append({
                    "id": annotation_id,
                    "image_id": img_id,
                    "category_id": 1,
                    "bbox": [x, y, w, h],
                    "area": area,
                    "iscrowd": 0
                })
                annotation_id += 1

    print("Формируем обучающую выборку (Train)...")
    process_split(train_frames, coco_train, train_dir)
    
    print("Формируем валидационную выборку (Val)...")
    process_split(val_frames, coco_val, val_dir)
    train_json_path = os.path.join(output_dataset_dir, "train_coco.json")
    val_json_path = os.path.join(output_dataset_dir, "val_coco.json")
    
    with open(train_json_path, 'w') as f:
        json.dump(coco_train, f, indent=4)
        
    with open(val_json_path, 'w') as f:
        json.dump(coco_val, f, indent=4)

    print(f"Готово! Датасет в формате COCO успешно создан в папке {output_dataset_dir}")
create_coco_dataset(
    frames_folder="../data/extracted_frames",  # Откуда брать чистые картинки
    annotations_path="../data/yolo_annotations.json", # Разметка от лучшей YOLOv8m
    output_dataset_dir="../data/coco_dataset"  # Куда сохранить готовый датасет
)

Загрузка разметки...
Всего кадров: 61. На обучение (Train): 48, На проверку (Val): 13
Формируем обучающую выборку (Train)...
Формируем валидационную выборку (Val)...
Готово! Датасет в формате COCO успешно создан в папке ../data/coco_dataset
